# Patent and Policy Citation Landscape

This notebook characterizes the external citation pathways of the ML-OPF
academic literature by comparing scholarly publications cited in patents
with those cited in policy documents.

The analysis covers:

1. coverage of patent and policy citations;
2. temporal evolution of the two citation pathways;
3. most frequently patent-cited publications;
4. most frequently policy-cited publications; and
5. leading journals, authors, and countries represented in each pathway.

In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

from pathlib import Path
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Project directories
# ---------------------------------------------------------

PROJECT_DIR = (
    Path.cwd()
    .resolve()
    .parent
)


DATA_DIR = (
    PROJECT_DIR
    / "data"
)

LDA_DATA_DIR = (
    DATA_DIR
    / "lda"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "output"
    / "patent_policy_citation_landscape"
)

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

TABLE_DIR = (
    OUTPUT_DIR
    / "tables"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LENS_FILE = (
    LDA_DATA_DIR
    / "lens_from_scopus_scholar.xlsx"
)

OVERTON_FILE = (
    LDA_DATA_DIR
    / "overton_from_scopus_scholar.xlsx"
)


print("INPUT FILES")
print("=" * 60)

print(
    "Lens   :",
    LENS_FILE
)

print(
    "Overton:",
    OVERTON_FILE
)

In [ ]:
# ---------------------------------------------------------
# Validate input files
# ---------------------------------------------------------

for file_path in [
    LENS_FILE,
    OVERTON_FILE,
]:

    if not file_path.exists():

        raise FileNotFoundError(
            f"Input file not found:\n"
            f"{file_path}"
        )


print(
    "Both input files found."
)

In [ ]:
# ---------------------------------------------------------
# Inspect workbook sheets
# ---------------------------------------------------------

lens_excel = pd.ExcelFile(
    LENS_FILE
)

overton_excel = pd.ExcelFile(
    OVERTON_FILE
)


print("LENS WORKBOOK")
print("=" * 50)

print(
    "Sheets:",
    lens_excel.sheet_names
)


print()
print("OVERTON WORKBOOK")
print("=" * 50)

print(
    "Sheets:",
    overton_excel.sheet_names
)

In [ ]:
# ---------------------------------------------------------
# Load patent-cited and policy-cited scholarly publications
# ---------------------------------------------------------

patent_cited = pd.read_excel(
    LENS_FILE,
    sheet_name=0,
)

policy_cited = pd.read_excel(
    OVERTON_FILE,
    sheet_name=0,
)


print("PATENT-CITED SCHOLARLY CORPUS")
print("=" * 60)

print(
    "Rows   :",
    f"{len(patent_cited):,}"
)

print(
    "Columns:",
    f"{len(patent_cited.columns):,}"
)

print()

for column in patent_cited.columns:

    print(
        " -",
        column
    )


print()
print("POLICY-CITED SCHOLARLY CORPUS")
print("=" * 60)

print(
    "Rows   :",
    f"{len(policy_cited):,}"
)

print(
    "Columns:",
    f"{len(policy_cited.columns):,}"
)

print()

for column in policy_cited.columns:

    print(
        " -",
        column
    )

In [ ]:
display(
    patent_cited.head()
)

display(
    policy_cited.head()
)

In [ ]:
# ---------------------------------------------------------
# Reload updated patent and policy datasets
# ---------------------------------------------------------

patent_cited = pd.read_excel(
    LENS_FILE,
    sheet_name=lens_excel.sheet_names[0],
)

policy_cited = pd.read_excel(
    OVERTON_FILE,
    sheet_name=overton_excel.sheet_names[0],
)


print("UPDATED EXTERNAL-CITATION DATASETS")
print("=" * 55)

print(
    "Patent-cited:",
    patent_cited.shape
)

print(
    "Policy-cited:",
    policy_cited.shape
)

print()

print("Patent-specific fields:")

for column in [
    "Patent citation count",
    "Citing patent family count",
]:

    print(
        f" - {column}:",
        column in patent_cited.columns
    )

In [ ]:
# ---------------------------------------------------------
# Normalize common bibliographic fields
# ---------------------------------------------------------

def normalize_doi(series):

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(
            r"^https?://(dx\.)?doi\.org/",
            "",
            regex=True,
        )
        .str.replace(
            r"^doi:\s*",
            "",
            regex=True,
        )
        .replace(
            "",
            np.nan,
        )
    )


# Patent/Lens
patent_cited["DOI_clean"] = normalize_doi(
    patent_cited["DOI"]
)

patent_cited["Year_clean"] = pd.to_numeric(
    patent_cited["Year"],
    errors="coerce",
)

patent_cited["Journal_clean"] = (
    patent_cited["Source title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

patent_cited["Title_clean"] = (
    patent_cited["Title"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Policy/Overton
policy_cited["DOI_clean"] = normalize_doi(
    policy_cited["DOI"]
)

policy_cited["Year_clean"] = pd.to_numeric(
    policy_cited["Year"],
    errors="coerce",
)

policy_cited["Journal_clean"] = (
    policy_cited["Journal"]
    .fillna("")
    .astype(str)
    .str.strip()
)

policy_cited["Title_clean"] = (
    policy_cited["Title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

policy_cited[
    "Policy_Citation_Count"
] = pd.to_numeric(
    policy_cited[
        "Policy citation count"
    ],
    errors="coerce",
)

In [ ]:
# ---------------------------------------------------------
# Normalize fields after reload
# ---------------------------------------------------------

patent_cited["DOI_clean"] = normalize_doi(
    patent_cited["DOI"]
)

policy_cited["DOI_clean"] = normalize_doi(
    policy_cited["DOI"]
)


patent_cited["Year_clean"] = pd.to_numeric(
    patent_cited["Year"],
    errors="coerce",
)

policy_cited["Year_clean"] = pd.to_numeric(
    policy_cited["Year"],
    errors="coerce",
)


patent_cited[
    "Patent_Citation_Count"
] = pd.to_numeric(
    patent_cited[
        "Patent citation count"
    ],
    errors="coerce",
)


patent_cited[
    "Patent_Family_Count"
] = pd.to_numeric(
    patent_cited[
        "Citing patent family count"
    ],
    errors="coerce",
)


policy_cited[
    "Policy_Citation_Count"
] = pd.to_numeric(
    policy_cited[
        "Policy citation count"
    ],
    errors="coerce",
)

In [ ]:
# ---------------------------------------------------------
# External-citation intensity
# ---------------------------------------------------------

external_citation_summary = pd.DataFrame(
    {
        "Indicator": [
            "Cited scholarly publications",
            "Total external citations",
            "Mean citations per cited publication",
            "Median citations per cited publication",
            "Maximum citations to one publication",
        ],

        "Patent": [
            len(patent_cited),
            patent_cited["Patent_Citation_Count"].sum(),
            patent_cited["Patent_Citation_Count"].mean(),
            patent_cited["Patent_Citation_Count"].median(),
            patent_cited["Patent_Citation_Count"].max(),
        ],

        "Policy": [
            len(policy_cited),
            policy_cited["Policy_Citation_Count"].sum(),
            policy_cited["Policy_Citation_Count"].mean(),
            policy_cited["Policy_Citation_Count"].median(),
            policy_cited["Policy_Citation_Count"].max(),
        ],
    }
)


display(
    external_citation_summary
)

In [ ]:
# ---------------------------------------------------------
# Load academic baseline
# ---------------------------------------------------------

SCOPUS_BASELINE_FILE = (
    DATA_DIR
    / "scopus"
    / "scopus_full.xlsx"
)


scopus_baseline = pd.read_excel(
    SCOPUS_BASELINE_FILE
)


scopus_baseline[
    "DOI_clean"
] = normalize_doi(
    scopus_baseline[
        "DOI"
    ]
)


scopus_baseline[
    "Year_clean"
] = pd.to_numeric(
    scopus_baseline[
        "Year"
    ],
    errors="coerce",
)


# Same analytical endpoint as the manuscript
MAX_ANALYSIS_YEAR = 2026


academic_baseline = (
    scopus_baseline[
        scopus_baseline[
            "Year_clean"
        ]
        .le(
            MAX_ANALYSIS_YEAR
        )
    ]
    .copy()
)


academic_dois = set(
    academic_baseline[
        "DOI_clean"
    ]
    .dropna()
)


print("ACADEMIC BASELINE")
print("=" * 50)

print(
    "Documents:",
    f"{len(academic_baseline):,}"
)

print(
    "Unique DOI:",
    f"{len(academic_dois):,}"
)

In [ ]:
# ---------------------------------------------------------
# External-citation coverage of academic literature
# ---------------------------------------------------------

patent_dois = set(
    patent_cited[
        "DOI_clean"
    ]
    .dropna()
)

policy_dois = set(
    policy_cited[
        "DOI_clean"
    ]
    .dropna()
)


n_academic = len(
    academic_dois
)

n_patent = len(
    patent_dois
    &
    academic_dois
)

n_policy = len(
    policy_dois
    &
    academic_dois
)

n_shared = len(
    patent_dois
    &
    policy_dois
    &
    academic_dois
)


patent_coverage = (
    100
    *
    n_patent
    /
    n_academic
)

policy_coverage = (
    100
    *
    n_policy
    /
    n_academic
)

dual_coverage = (
    100
    *
    n_shared
    /
    n_academic
)


coverage_summary = pd.DataFrame(
    {
        "Pathway": [
            "Patent-cited",
            "Policy-cited",
            "Dual-pathway",
        ],

        "Publications": [
            n_patent,
            n_policy,
            n_shared,
        ],

        "Academic_Coverage_Percent": [
            patent_coverage,
            policy_coverage,
            dual_coverage,
        ],
    }
)


coverage_summary[
    "Academic_Coverage_Percent"
] = (
    coverage_summary[
        "Academic_Coverage_Percent"
    ]
    .round(2)
)


display(
    coverage_summary
)

In [ ]:
# ---------------------------------------------------------
# Patent- and policy-citation intensity
# ---------------------------------------------------------

patent_citation_stats = {
    "Total": (
        patent_cited[
            "Patent_Citation_Count"
        ].sum()
    ),
    "Mean": (
        patent_cited[
            "Patent_Citation_Count"
        ].mean()
    ),
    "Median": (
        patent_cited[
            "Patent_Citation_Count"
        ].median()
    ),
    "Maximum": (
        patent_cited[
            "Patent_Citation_Count"
        ].max()
    ),
}


policy_citation_stats = {
    "Total": (
        policy_cited[
            "Policy_Citation_Count"
        ].sum()
    ),
    "Mean": (
        policy_cited[
            "Policy_Citation_Count"
        ].mean()
    ),
    "Median": (
        policy_cited[
            "Policy_Citation_Count"
        ].median()
    ),
    "Maximum": (
        policy_cited[
            "Policy_Citation_Count"
        ].max()
    ),
}


citation_intensity_summary = pd.DataFrame(
    {
        "Metric": [
            "Total external citations",
            "Mean citations per cited publication",
            "Median citations per cited publication",
            "Maximum citations to one publication",
        ],

        "Patent": [
            patent_citation_stats["Total"],
            patent_citation_stats["Mean"],
            patent_citation_stats["Median"],
            patent_citation_stats["Maximum"],
        ],

        "Policy": [
            policy_citation_stats["Total"],
            policy_citation_stats["Mean"],
            policy_citation_stats["Median"],
            policy_citation_stats["Maximum"],
        ],
    }
)


display(
    citation_intensity_summary
)

In [ ]:
# ---------------------------------------------------------
# Patent-family citation statistics
# ---------------------------------------------------------

patent_family_summary = pd.DataFrame(
    {
        "Metric": [
            "Total citing patent families",
            "Mean families per cited publication",
            "Median families per cited publication",
            "Maximum families citing one publication",
        ],

        "Value": [
            patent_cited[
                "Patent_Family_Count"
            ].sum(),

            patent_cited[
                "Patent_Family_Count"
            ].mean(),

            patent_cited[
                "Patent_Family_Count"
            ].median(),

            patent_cited[
                "Patent_Family_Count"
            ].max(),
        ],
    }
)


display(
    patent_family_summary
)

In [ ]:
# ---------------------------------------------------------
# Integrated patent-policy citation landscape
# ---------------------------------------------------------

coverage_landscape = pd.DataFrame(
    {
        "Indicator": [
            "Cited publications",
            "Academic coverage (%)",
            "Pathway-exclusive publications",
            "Shared publications",
            "Share also present in other pathway (%)",
            "Total external citations",
            "Mean citations per cited publication",
            "Median citations per cited publication",
            "Maximum citations to one publication",
        ],

        "Patent": [
            778,
            4.93,
            695,
            83,
            10.67,
            patent_citation_stats["Total"],
            patent_citation_stats["Mean"],
            patent_citation_stats["Median"],
            patent_citation_stats["Maximum"],
        ],

        "Policy": [
            392,
            2.48,
            309,
            83,
            21.17,
            policy_citation_stats["Total"],
            policy_citation_stats["Mean"],
            policy_citation_stats["Median"],
            policy_citation_stats["Maximum"],
        ],
    }
)


display(
    coverage_landscape
)

In [ ]:
# ---------------------------------------------------------
# Patent-policy external citation pathways
# ---------------------------------------------------------

# ---------------------------------------------------------
# Reconstruct pathway counts from DOI sets
# ---------------------------------------------------------

patent_only_dois = (
    patent_dois
    - policy_dois
)

policy_only_dois = (
    policy_dois
    - patent_dois
)

shared_dois = (
    patent_dois
    & policy_dois
)


n_patent_only = len(
    patent_only_dois
)

n_policy_only = len(
    policy_only_dois
)

n_shared = len(
    shared_dois
)

n_union = (
    n_patent_only
    + n_policy_only
    + n_shared
)


# ---------------------------------------------------------
# Build pathway summary
# ---------------------------------------------------------

citation_pathway_overlap = pd.DataFrame(
    {
        "Pathway": [
            "Patent only",
            "Policy only",
            "Patent and policy",
        ],

        "Documents": [
            n_patent_only,
            n_policy_only,
            n_shared,
        ],
    }
)


citation_pathway_overlap[
    "Percent_of_external_cited"
] = (
    100
    * citation_pathway_overlap[
        "Documents"
    ]
    / n_union
)


print(
    "PATENT-POLICY CITATION PATHWAYS"
)

print(
    "=" * 50
)

print(
    "Patent only      :",
    f"{n_patent_only:,}"
)

print(
    "Policy only      :",
    f"{n_policy_only:,}"
)

print(
    "Patent and policy:",
    f"{n_shared:,}"
)

print(
    "Union            :",
    f"{n_union:,}"
)

print()


display(
    citation_pathway_overlap
)


# ---------------------------------------------------------
# Prepare plot
# ---------------------------------------------------------

pathway_plot = (
    citation_pathway_overlap
    .copy()
)


pathway_plot[
    "Percent_of_external_cited"
] = (
    pathway_plot[
        "Percent_of_external_cited"
    ]
    .astype(float)
)


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(
        8.0,
        4.8,
    )
)


bars = ax.barh(
    pathway_plot[
        "Pathway"
    ],
    pathway_plot[
        "Documents"
    ],
)


# ---------------------------------------------------------
# Labels
# ---------------------------------------------------------

for bar, (_, row) in zip(
    bars,
    pathway_plot.iterrows(),
):

    ax.text(
        bar.get_width() + 12,

        bar.get_y()
        + bar.get_height() / 2,

        (
            f"{int(row['Documents']):,} "
            f"({row['Percent_of_external_cited']:.1f}%)"
        ),

        va="center",

        fontsize=10,

        fontweight="bold",
    )


# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------

ax.set_xlabel(
    "Number of scholarly publications"
)


ax.set_ylabel(
    ""
)


ax.set_title(
    "Patent and Policy Citation Pathways"
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


ax.grid(
    axis="x",
    alpha=0.20,
)


ax.set_axisbelow(
    True
)


ax.set_xlim(
    0,
    pathway_plot[
        "Documents"
    ].max()
    * 1.20,
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

fig.tight_layout()


overlap_figure = (
    FIGURE_DIR
    / "patent_policy_pathway_overlap.pdf"
)


fig.savefig(
    overlap_figure,
    format="pdf",
    bbox_inches="tight",
)


print(
    "Saved:",
    overlap_figure
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Venn diagram: patent- and policy-cited publications
# ---------------------------------------------------------

from matplotlib_venn import venn2
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Counts
# ---------------------------------------------------------

n_patent_only = len(
    patent_only_dois
)

n_policy_only = len(
    policy_only_dois
)

n_shared = len(
    shared_dois
)


n_patent_total = (
    n_patent_only
    + n_shared
)

n_policy_total = (
    n_policy_only
    + n_shared
)

n_union = (
    n_patent_only
    + n_policy_only
    + n_shared
)


# ---------------------------------------------------------
# Percentages of all unique externally cited publications
# ---------------------------------------------------------

patent_only_pct = (
    100
    * n_patent_only
    / n_union
)

policy_only_pct = (
    100
    * n_policy_only
    / n_union
)

shared_pct = (
    100
    * n_shared
    / n_union
)


# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print("PATENT-POLICY CITATION OVERLAP")
print("=" * 50)

print(
    "Patent-cited total :",
    f"{n_patent_total:,}"
)

print(
    "Policy-cited total :",
    f"{n_policy_total:,}"
)

print(
    "Unique union       :",
    f"{n_union:,}"
)

print(
    "Patent-cited only  :",
    f"{n_patent_only:,} ({patent_only_pct:.1f}%)"
)

print(
    "Policy-cited only  :",
    f"{n_policy_only:,} ({policy_only_pct:.1f}%)"
)

print(
    "Shared             :",
    f"{n_shared:,} ({shared_pct:.1f}%)"
)


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)


venn = venn2(
    subsets=(
        n_patent_only,
        n_policy_only,
        n_shared,
    ),
    set_labels=(
        f"Patent-cited\n($n={n_patent_total:,}$)",
        f"Policy-cited\n($n={n_policy_total:,}$)",
    ),
    ax=ax,
)


# ---------------------------------------------------------
# Region labels: count + percentage
# Percentage denominator = unique external-citation union
# ---------------------------------------------------------

region_labels = {
    "10": (
        f"{n_patent_only:,} "
        f"({patent_only_pct:.1f}%)\n"
        "Patent-cited only"
    ),

    "01": (
        f"{n_policy_only:,} "
        f"({policy_only_pct:.1f}%)\n"
        "Policy-cited only"
    ),

    "11": (
        f"{n_shared:,} "
        f"({shared_pct:.1f}%)\n"
        "Shared"
    ),
}


for region_id, region_text in region_labels.items():

    label = venn.get_label_by_id(
        region_id
    )

    if label is not None:

        label.set_text(
            region_text
        )

        label.set_fontsize(
            10.5
        )

        label.set_fontweight(
            "bold"
        )

        label.set_ha(
            "center"
        )

        label.set_va(
            "center"
        )


# ---------------------------------------------------------
# Set-label formatting
# ---------------------------------------------------------

for label in venn.set_labels:

    if label is not None:

        label.set_fontsize(
            11
        )

        label.set_fontweight(
            "bold"
        )


# ---------------------------------------------------------
# Title
# ---------------------------------------------------------

ax.set_title(
    "Overlap Between Patent- and Policy-Cited ML-OPF Publications",
    fontsize=13,
    fontweight="bold",
    pad=18,
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

fig.tight_layout()


venn_figure = (
    FIGURE_DIR
    / "patent_policy_citation_overlap.pdf"
)


fig.savefig(
    venn_figure,
    format="pdf",
    bbox_inches="tight",
)


print()
print(
    "Saved:",
    venn_figure
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Temporal evolution of external citations
# ---------------------------------------------------------

MAX_ANALYSIS_YEAR = 2026


patent_temporal = (
    patent_cited
    .loc[
        patent_cited[
            "Year_clean"
        ].le(
            MAX_ANALYSIS_YEAR
        )
    ]
    .groupby(
        "Year_clean"
    )
    .agg(
        Patent_Cited_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Patent_Citations=(
            "Patent_Citation_Count",
            "sum",
        ),
        Mean_Patent_Citations=(
            "Patent_Citation_Count",
            "mean",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "Year_clean":
                "Year"
        }
    )
)


policy_temporal = (
    policy_cited
    .loc[
        policy_cited[
            "Year_clean"
        ].le(
            MAX_ANALYSIS_YEAR
        )
    ]
    .groupby(
        "Year_clean"
    )
    .agg(
        Policy_Cited_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Policy_Citations=(
            "Policy_Citation_Count",
            "sum",
        ),
        Mean_Policy_Citations=(
            "Policy_Citation_Count",
            "mean",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "Year_clean":
                "Year"
        }
    )
)


external_temporal = (
    patent_temporal
    .merge(
        policy_temporal,
        on="Year",
        how="outer",
    )
    .sort_values(
        "Year"
    )
    .reset_index(
        drop=True
    )
)


display(
    external_temporal
)

In [ ]:
# ---------------------------------------------------------
# Most frequently patent-cited publications
# ---------------------------------------------------------

TOP_N_PUBLICATIONS = 10


top_patent_publications = (
    patent_cited[
        [
            "Title",
            "Authors",
            "Year",
            "Source title",
            "DOI",
            "Patent_Citation_Count",
            "Patent_Family_Count",
            "Cited by",
        ]
    ]
    .sort_values(
        [
            "Patent_Citation_Count",
            "Patent_Family_Count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        TOP_N_PUBLICATIONS
    )
    .reset_index(
        drop=True
    )
)


top_patent_publications.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(top_patent_publications) + 1,
    ),
)


display(
    top_patent_publications
)

In [ ]:
# ---------------------------------------------------------
# Most frequently policy-cited publications
# ---------------------------------------------------------

top_policy_publications = (
    policy_cited[
        [
            "Title",
            "Authors",
            "Year",
            "Journal",
            "DOI",
            "Policy_Citation_Count",
            "Cited by",
        ]
    ]
    .sort_values(
        "Policy_Citation_Count",
        ascending=False,
    )
    .head(
        TOP_N_PUBLICATIONS
    )
    .reset_index(
        drop=True
    )
)


top_policy_publications.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(top_policy_publications) + 1,
    ),
)


display(
    top_policy_publications
)

In [ ]:
# ---------------------------------------------------------
# Journal representation across external-citation pathways
# ---------------------------------------------------------

patent_journals = (
    patent_cited
    .groupby(
        "Source title"
    )
    .agg(
        Patent_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Patent_Citations=(
            "Patent_Citation_Count",
            "sum",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "Source title":
                "Journal"
        }
    )
)


policy_journals = (
    policy_cited
    .groupby(
        "Journal"
    )
    .agg(
        Policy_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Policy_Citations=(
            "Policy_Citation_Count",
            "sum",
        ),
    )
    .reset_index()
)


journal_comparison = (
    patent_journals
    .merge(
        policy_journals,
        on="Journal",
        how="outer",
    )
    .fillna(
        0
    )
)


journal_comparison[
    "Total_External_Publications"
] = (
    journal_comparison[
        "Patent_Publications"
    ]
    +
    journal_comparison[
        "Policy_Publications"
    ]
)


journal_comparison = (
    journal_comparison
    .sort_values(
        "Total_External_Publications",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    journal_comparison.head(20)
)

In [ ]:
# ---------------------------------------------------------
# Temporal evolution of patent- and policy-cited publications
# ---------------------------------------------------------

TEMPORAL_START_YEAR = 2015
TEMPORAL_END_YEAR = 2026


temporal_plot = (
    external_temporal
    .loc[
        lambda df:
            df["Year"].between(
                TEMPORAL_START_YEAR,
                TEMPORAL_END_YEAR,
            )
    ]
    .copy()
)


# Years without observations represent zero cited publications.
for column in [
    "Patent_Cited_Publications",
    "Policy_Cited_Publications",
]:

    temporal_plot[
        column
    ] = (
        temporal_plot[
            column
        ]
        .fillna(0)
    )


fig, ax = plt.subplots(
    figsize=(8.5, 5.0)
)


ax.plot(
    temporal_plot["Year"],
    temporal_plot["Patent_Cited_Publications"],
    marker="o",
    linewidth=2,
    label="Patent-cited publications",
)


ax.plot(
    temporal_plot["Year"],
    temporal_plot["Policy_Cited_Publications"],
    marker="s",
    linewidth=2,
    label="Policy-cited publications",
)


ax.set_xlabel(
    "Publication year"
)

ax.set_ylabel(
    "Number of externally cited publications"
)


ax.set_xticks(
    temporal_plot["Year"]
)


ax.legend(
    frameon=True
)


ax.grid(
    alpha=0.25
)


fig.tight_layout()


temporal_figure = (
    FIGURE_DIR
    / "patent_policy_temporal_evolution.pdf"
)


fig.savefig(
    temporal_figure,
    format="pdf",
    bbox_inches="tight",
)


print(
    "Saved:",
    temporal_figure
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Compact top patent-cited table
# ---------------------------------------------------------

top_patent_table = (
    top_patent_publications[
        [
            "Rank",
            "Title",
            "Year",
            "Source title",
            "Patent_Citation_Count",
            "Patent_Family_Count",
        ]
    ]
    .copy()
    .rename(
        columns={
            "Source title":
                "Journal",

            "Patent_Citation_Count":
                "Patent citations",

            "Patent_Family_Count":
                "Citing patent families",
        }
    )
)


display(
    top_patent_table
)

# ---------------------------------------------------------
# Save top patent-cited publications
# ---------------------------------------------------------

top_patent_table_file = (
    TABLE_DIR
    / "top_patent_cited_publications.csv"
)


top_patent_table.to_csv(
    top_patent_table_file,
    index=False,
)


print(
    "Saved:",
    top_patent_table_file
)

In [ ]:
# ---------------------------------------------------------
# Compact top policy-cited table
# ---------------------------------------------------------

top_policy_table = (
    top_policy_publications[
        [
            "Rank",
            "Title",
            "Year",
            "Journal",
            "Policy_Citation_Count",
        ]
    ]
    .copy()
    .rename(
        columns={
            "Policy_Citation_Count":
                "Policy citations",
        }
    )
)


display(
    top_policy_table
)

# ---------------------------------------------------------
# Save top policy-cited publications
# ---------------------------------------------------------

top_policy_table_file = (
    TABLE_DIR
    / "top_policy_cited_publications.csv"
)


top_policy_table.to_csv(
    top_policy_table_file,
    index=False,
)


print(
    "Saved:",
    top_policy_table_file
)

In [ ]:
# ---------------------------------------------------------
# Comparative journal representation
# ---------------------------------------------------------

journal_comparison[
    "Patent_Publications"
] = (
    journal_comparison[
        "Patent_Publications"
    ]
    .astype(int)
)

journal_comparison[
    "Policy_Publications"
] = (
    journal_comparison[
        "Policy_Publications"
    ]
    .astype(int)
)

journal_comparison[
    "Patent_Citations"
] = (
    journal_comparison[
        "Patent_Citations"
    ]
    .astype(int)
)

journal_comparison[
    "Policy_Citations"
] = (
    journal_comparison[
        "Policy_Citations"
    ]
    .astype(int)
)


journal_comparison[
    "Patent_Rank"
] = (
    journal_comparison[
        "Patent_Publications"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


journal_comparison[
    "Policy_Rank"
] = (
    journal_comparison[
        "Policy_Publications"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


journal_comparison = (
    journal_comparison
    .sort_values(
        [
            "Patent_Publications",
            "Policy_Publications",
        ],
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    journal_comparison.head(20)
)

In [ ]:
# ---------------------------------------------------------
# Parse individual Scopus authors
# ---------------------------------------------------------

def parse_scopus_authors(value):

    if pd.isna(value):
        return []

    return [
        author.strip()
        for author in str(value).split(";")
        if author.strip()
    ]

# ---------------------------------------------------------
# Patent-cited authors
# ---------------------------------------------------------

patent_author_data = (
    patent_cited[
        [
            "DOI_clean",
            "Author full names",
            "Patent_Citation_Count",
        ]
    ]
    .copy()
)


patent_author_data[
    "Author"
] = (
    patent_author_data[
        "Author full names"
    ]
    .apply(
        parse_scopus_authors
    )
)


patent_author_data = (
    patent_author_data
    .explode(
        "Author"
    )
    .dropna(
        subset=[
            "Author"
        ]
    )
)    

# ---------------------------------------------------------
# Policy-cited authors
# ---------------------------------------------------------

policy_author_data = (
    policy_cited[
        [
            "DOI_clean",
            "Author full names",
            "Policy_Citation_Count",
        ]
    ]
    .copy()
)


policy_author_data[
    "Author"
] = (
    policy_author_data[
        "Author full names"
    ]
    .apply(
        parse_scopus_authors
    )
)


policy_author_data = (
    policy_author_data
    .explode(
        "Author"
    )
    .dropna(
        subset=[
            "Author"
        ]
    )
)

# ---------------------------------------------------------
# Author-level external citation representation
# ---------------------------------------------------------

patent_authors = (
    patent_author_data
    .groupby(
        "Author"
    )
    .agg(
        Patent_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Patent_Citations=(
            "Patent_Citation_Count",
            "sum",
        ),
    )
    .reset_index()
)


policy_authors = (
    policy_author_data
    .groupby(
        "Author"
    )
    .agg(
        Policy_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Policy_Citations=(
            "Policy_Citation_Count",
            "sum",
        ),
    )
    .reset_index()
)


author_comparison = (
    patent_authors
    .merge(
        policy_authors,
        on="Author",
        how="outer",
    )
    .fillna(0)
)


for column in [
    "Patent_Publications",
    "Patent_Citations",
    "Policy_Publications",
    "Policy_Citations",
]:

    author_comparison[
        column
    ] = (
        author_comparison[
            column
        ]
        .astype(int)
    )


display(
    author_comparison
    .sort_values(
        "Patent_Publications",
        ascending=False,
    )
    .head(20)
)

display(
    author_comparison
    .sort_values(
        "Policy_Publications",
        ascending=False,
    )
    .head(20)
)

In [ ]:
# ---------------------------------------------------------
# Country normalization dictionary
# ---------------------------------------------------------

COUNTRY_ALIASES = {

    "United States": "United States",
    "USA": "United States",

    "United Kingdom": "United Kingdom",
    "UK": "United Kingdom",

    "China": "China",
    "Australia": "Australia",
    "Malaysia": "Malaysia",
    "Algeria": "Algeria",
    "Cyprus": "Cyprus",
    "South Korea": "South Korea",
    "Korea": "South Korea",
    "Canada": "Canada",
    "Poland": "Poland",
    "Italy": "Italy",

    "Netherlands": "Netherlands",
    "Belgium": "Belgium",
    "Germany": "Germany",

    "India": "India",
    "Pakistan": "Pakistan",
    "Iraq": "Iraq",
    "Uzbekistan": "Uzbekistan",
    "South Africa": "South Africa",
    "Jordan": "Jordan",
    "Ukraine": "Ukraine",

    "Saudi Arabia": "Saudi Arabia",
    "France": "France",
    "Spain": "Spain",
    "Portugal": "Portugal",
    "Sweden": "Sweden",
    "Norway": "Norway",
    "Denmark": "Denmark",
    "Finland": "Finland",
    "Switzerland": "Switzerland",
    "Austria": "Austria",
    "Greece": "Greece",
    "Turkey": "Turkey",
    "Türkiye": "Turkey",

    "Japan": "Japan",
    "Singapore": "Singapore",
    "Taiwan": "Taiwan",
    "Hong Kong": "Hong Kong",

    "Brazil": "Brazil",
    "Mexico": "Mexico",
    "Chile": "Chile",
    "Argentina": "Argentina",

    "Iran": "Iran",
    "United Arab Emirates": "United Arab Emirates",
    "Qatar": "Qatar",
    "Egypt": "Egypt",
}

In [ ]:
# ---------------------------------------------------------
# Extract countries from affiliation strings
# ---------------------------------------------------------

def extract_countries(value):

    if pd.isna(value):
        return []

    text = str(value)

    countries = set()

    for alias, canonical in COUNTRY_ALIASES.items():

        if alias.lower() in text.lower():

            countries.add(
                canonical
            )

    return sorted(
        countries
    )

In [ ]:
# ---------------------------------------------------------
# Patent-cited country representation
# ---------------------------------------------------------

patent_country_data = (
    patent_cited[
        [
            "DOI_clean",
            "Affiliations",
            "Patent_Citation_Count",
        ]
    ]
    .copy()
)


patent_country_data[
    "Country"
] = (
    patent_country_data[
        "Affiliations"
    ]
    .apply(
        extract_countries
    )
)


patent_country_data = (
    patent_country_data
    .explode(
        "Country"
    )
    .dropna(
        subset=[
            "Country"
        ]
    )
)

In [ ]:
# ---------------------------------------------------------
# Policy-cited country representation
# ---------------------------------------------------------

policy_country_data = (
    policy_cited[
        [
            "DOI_clean",
            "Affiliations",
            "Policy_Citation_Count",
        ]
    ]
    .copy()
)


policy_country_data[
    "Country"
] = (
    policy_country_data[
        "Affiliations"
    ]
    .apply(
        extract_countries
    )
)


policy_country_data = (
    policy_country_data
    .explode(
        "Country"
    )
    .dropna(
        subset=[
            "Country"
        ]
    )
)

In [ ]:
patent_countries = (
    patent_country_data
    .groupby(
        "Country"
    )
    .agg(
        Patent_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Patent_Citations=(
            "Patent_Citation_Count",
            "sum",
        ),
    )
    .reset_index()
)


policy_countries = (
    policy_country_data
    .groupby(
        "Country"
    )
    .agg(
        Policy_Publications=(
            "DOI_clean",
            "nunique",
        ),
        Policy_Citations=(
            "Policy_Citation_Count",
            "sum",
        ),
    )
    .reset_index()
)


country_comparison = (
    patent_countries
    .merge(
        policy_countries,
        on="Country",
        how="outer",
    )
    .fillna(0)
)


for column in [
    "Patent_Publications",
    "Patent_Citations",
    "Policy_Publications",
    "Policy_Citations",
]:

    country_comparison[
        column
    ] = (
        country_comparison[
            column
        ]
        .astype(int)
    )


display(
    country_comparison
    .sort_values(
        "Patent_Publications",
        ascending=False,
    )
    .head(20)
)

In [ ]:
display(
    country_comparison
    .sort_values(
        "Policy_Publications",
        ascending=False,
    )
    .head(20)
)

In [ ]:
# ---------------------------------------------------------
# Leading authors in the patent-citation pathway
# ---------------------------------------------------------

top_patent_authors = (
    author_comparison
    .sort_values(
        [
            "Patent_Publications",
            "Patent_Citations",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(10)
    .reset_index(drop=True)
)


top_patent_authors.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(top_patent_authors) + 1,
    ),
)


display(
    top_patent_authors
)

In [ ]:
# ---------------------------------------------------------
# Leading authors in the policy-citation pathway
# ---------------------------------------------------------

top_policy_authors = (
    author_comparison
    .sort_values(
        [
            "Policy_Publications",
            "Policy_Citations",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(10)
    .reset_index(drop=True)
)


top_policy_authors.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(top_policy_authors) + 1,
    ),
)


display(
    top_policy_authors
)

In [ ]:
# ---------------------------------------------------------
# Top journals by pathway
# ---------------------------------------------------------

top_patent_journals = (
    journal_comparison
    .sort_values(
        [
            "Patent_Publications",
            "Patent_Citations",
        ],
        ascending=False,
    )
    .head(5)
    [
        [
            "Journal",
            "Patent_Publications",
            "Patent_Citations",
        ]
    ]
    .reset_index(drop=True)
)


top_policy_journals = (
    journal_comparison
    .sort_values(
        [
            "Policy_Publications",
            "Policy_Citations",
        ],
        ascending=False,
    )
    .head(5)
    [
        [
            "Journal",
            "Policy_Publications",
            "Policy_Citations",
        ]
    ]
    .reset_index(drop=True)
)


display(top_patent_journals)
display(top_policy_journals)

In [ ]:
# ---------------------------------------------------------
# Top countries by pathway
# ---------------------------------------------------------

top_patent_countries = (
    country_comparison
    .sort_values(
        [
            "Patent_Publications",
            "Patent_Citations",
        ],
        ascending=False,
    )
    .head(5)
    [
        [
            "Country",
            "Patent_Publications",
            "Patent_Citations",
        ]
    ]
    .reset_index(drop=True)
)


top_policy_countries = (
    country_comparison
    .sort_values(
        [
            "Policy_Publications",
            "Policy_Citations",
        ],
        ascending=False,
    )
    .head(5)
    [
        [
            "Country",
            "Policy_Publications",
            "Policy_Citations",
        ]
    ]
    .reset_index(drop=True)
)


display(top_patent_countries)
display(top_policy_countries)

In [ ]:
# ---------------------------------------------------------
# Country pathway specialization
# ---------------------------------------------------------

EPSILON = 0.5

N_PATENT = len(
    patent_cited
)

N_POLICY = len(
    policy_cited
)


country_comparison[
    "Patent_Share"
] = (
    country_comparison[
        "Patent_Publications"
    ]
    /
    N_PATENT
)


country_comparison[
    "Policy_Share"
] = (
    country_comparison[
        "Policy_Publications"
    ]
    /
    N_POLICY
)


country_comparison[
    "Pathway_Specialization"
] = np.log2(
    (
        country_comparison[
            "Patent_Publications"
        ]
        + EPSILON
    )
    /
    (
        N_PATENT
        + EPSILON
    )
    /
    (
        (
            country_comparison[
                "Policy_Publications"
            ]
            + EPSILON
        )
        /
        (
            N_POLICY
            + EPSILON
        )
    )
)


display(
    country_comparison
    .sort_values(
        "Pathway_Specialization",
        ascending=False,
    )
    [
        [
            "Country",
            "Patent_Publications",
            "Policy_Publications",
            "Pathway_Specialization",
        ]
    ]
    .head(15)
)

In [ ]:
# ---------------------------------------------------------
# Top authors by patent-cited publications
# ---------------------------------------------------------

top_patent_authors = (
    author_comparison
    .sort_values(
        [
            "Patent_Publications",
            "Patent_Citations",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(5)
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Top authors by policy-cited publications
# ---------------------------------------------------------

top_policy_authors = (
    author_comparison
    .sort_values(
        [
            "Policy_Publications",
            "Policy_Citations",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(5)
    .reset_index(drop=True)
)


display(
    top_patent_authors
)

display(
    top_policy_authors
)

In [ ]:
# ---------------------------------------------------------
# Comparative leading-journal table
# ---------------------------------------------------------

TOP_N = 5


leading_journals_table = pd.DataFrame(
    {
        "Rank": np.arange(
            1,
            TOP_N + 1,
        ),

        "Patent_Journal": (
            top_patent_journals[
                "Journal"
            ]
            .head(TOP_N)
            .values
        ),

        "Patent_Publications": (
            top_patent_journals[
                "Patent_Publications"
            ]
            .head(TOP_N)
            .values
        ),

        "Patent_Citations": (
            top_patent_journals[
                "Patent_Citations"
            ]
            .head(TOP_N)
            .values
        ),

        "Policy_Journal": (
            top_policy_journals[
                "Journal"
            ]
            .head(TOP_N)
            .values
        ),

        "Policy_Publications": (
            top_policy_journals[
                "Policy_Publications"
            ]
            .head(TOP_N)
            .values
        ),

        "Policy_Citations": (
            top_policy_journals[
                "Policy_Citations"
            ]
            .head(TOP_N)
            .values
        ),
    }
)


display(
    leading_journals_table
)

In [ ]:
# ---------------------------------------------------------
# Generic external-citation donut timeline
# ---------------------------------------------------------

def plot_timeline_pies(
    df,
    x_col,
    hue_col,
    y_col=None,
    palette=None,
    donut_scale=0.35,
    x_spacing=3.4,
    figsize=(16, 6.5),
    title=None,
    y_label="Citation events",
    legend_title=None,
    output_file=None,
):

    # -----------------------------------------------------
    # Aggregate
    # -----------------------------------------------------

    if y_col is None:

        grouped = (
            df
            .groupby(
                [
                    x_col,
                    hue_col,
                ]
            )
            .size()
            .reset_index(
                name="__count__"
            )
        )

        value_col = "__count__"

    else:

        grouped = (
            df
            .groupby(
                [
                    x_col,
                    hue_col,
                ]
            )[
                y_col
            ]
            .sum()
            .reset_index()
        )

        value_col = y_col


    pivot_df = (
        grouped
        .pivot(
            index=x_col,
            columns=hue_col,
            values=value_col,
        )
        .fillna(0)
        .sort_index()
    )


    raw_x_values = (
        pivot_df
        .index
        .values
    )


    unique_hues = (
        pivot_df
        .columns
        .tolist()
    )


    # -----------------------------------------------------
    # X coordinates
    # -----------------------------------------------------

    x_coords = (
        np.arange(
            len(raw_x_values)
        )
        * x_spacing
    )


    x_map = dict(
        zip(
            raw_x_values,
            x_coords,
        )
    )


    # -----------------------------------------------------
    # Colors
    # -----------------------------------------------------

    if palette is None:

        cmap = plt.get_cmap(
            "tab10"
        )

        hue_colors = {
            hue:
                cmap(
                    i % 10
                )
            for i, hue
            in enumerate(
                unique_hues
            )
        }

    elif isinstance(
        palette,
        list,
    ):

        hue_colors = {
            hue:
                palette[
                    i % len(
                        palette
                    )
                ]
            for i, hue
            in enumerate(
                unique_hues
            )
        }

    else:

        hue_colors = palette


    # -----------------------------------------------------
    # Annual totals
    # -----------------------------------------------------

    annual_totals = (
        pivot_df
        .sum(
            axis=1
        )
    )


    # -----------------------------------------------------
    # Create donut images
    # -----------------------------------------------------

    pie_images = {}


    for x_val in raw_x_values:

        data_row = (
            pivot_df
            .loc[
                x_val
            ]
        )

        total_val = (
            annual_totals
            .loc[
                x_val
            ]
        )


        if total_val == 0:

            continue


        fig_pie, ax_pie = plt.subplots(
            figsize=(
                2.2,
                2.2,
            ),
            facecolor="none",
        )


        ax_pie.set_facecolor(
            "none"
        )


        ax_pie.pie(
            data_row.values,
            colors=[
                hue_colors[
                    hue
                ]
                for hue
                in unique_hues
            ],
            startangle=90,
            counterclock=False,
            wedgeprops={
                "width": 0.38,
                "edgecolor": "#0F172A",
                "linewidth": 0.8,
            },
        )


        ax_pie.text(
            0,
            0,
            f"{int(total_val):,}",
            ha="center",
            va="center",
            fontsize=23,
            fontweight="bold",
            color="#0F172A",
            bbox={
                "boxstyle":
                    "circle,pad=0.25",

                "facecolor":
                    "white",

                "edgecolor":
                    "#CBD5E1",

                "linewidth":
                    0.8,
            },
        )


        ax_pie.axis(
            "equal"
        )


        buffer = io.BytesIO()


        fig_pie.savefig(
            buffer,
            format="png",
            transparent=True,
            bbox_inches="tight",
            pad_inches=0.01,
            dpi=140,
        )


        buffer.seek(
            0
        )


        image_array = plt.imread(
            buffer
        )


        pie_images[
            x_val
        ] = OffsetImage(
            image_array,
            zoom=donut_scale,
        )


        plt.close(
            fig_pie
        )


    # -----------------------------------------------------
    # Main timeline
    # -----------------------------------------------------

    fig, ax = plt.subplots(
        figsize=figsize,
        facecolor="white",
        dpi=300,
    )


    plot_x = [
        x_map[
            value
        ]
        for value
        in raw_x_values
    ]


    plot_y = (
        annual_totals
        .values
    )


    ax.plot(
        plot_x,
        plot_y,
        linestyle="--",
        linewidth=1.6,
        zorder=2,
    )


    max_total = (
        annual_totals.max()
        if annual_totals.max() > 0
        else 10
    )


    ax.set_xlim(
        min(plot_x)
        - x_spacing * 0.8,

        max(plot_x)
        + x_spacing * 0.8,
    )


    ax.set_ylim(
        -1,
        max_total
        + max_total * 0.18,
    )


    # -----------------------------------------------------
    # Place donuts
    # -----------------------------------------------------

    for x_val in raw_x_values:

        total_val = (
            annual_totals
            .loc[
                x_val
            ]
        )


        if (
            total_val == 0
            or
            x_val not in pie_images
        ):

            continue


        annotation = AnnotationBbox(
            pie_images[
                x_val
            ],
            (
                x_map[
                    x_val
                ],
                total_val,
            ),
            frameon=False,
            pad=0,
            zorder=3,
        )


        ax.add_artist(
            annotation
        )


    # -----------------------------------------------------
    # Axes
    # -----------------------------------------------------

    ax.grid(
        axis="both",
        linestyle=":",
        alpha=0.7,
        color="#CBD5E1",
        zorder=0,
    )


    ax.set_xticks(
        plot_x
    )


    ax.set_xticklabels(
        [
            f"{int(value)}"
            if pd.notna(value)
            else ""
            for value
            in raw_x_values
        ],
        fontsize=11,
        fontweight="bold",
    )


    ax.set_ylabel(
        y_label,
        fontsize=14,
        fontweight="bold",
    )


    ax.set_xlabel(
        "Citation year",
        fontsize=14,
        fontweight="bold",
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    # -----------------------------------------------------
    # Legend
    # -----------------------------------------------------

    legend_handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=hue_colors[
                hue
            ],
            label=str(
                hue
            ),
        )
        for hue
        in unique_hues
    ]


    ax.legend(
        handles=legend_handles,
        title=(
            legend_title
            if legend_title
            else str(
                hue_col
            )
        ),
        bbox_to_anchor=(
            1.02,
            1.0,
        ),
        loc="upper left",
        frameon=True,
    )


    # -----------------------------------------------------
    # Title
    # -----------------------------------------------------

    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=18,
        loc="left",
    )


    fig.subplots_adjust(
        left=0.07,
        right=0.84,
        top=0.90,
        bottom=0.12,
    )


    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------

    if output_file is not None:

        fig.savefig(
            output_file,
            format="pdf",
            dpi=300,
            bbox_inches="tight",
        )


        print(
            "Saved:",
            output_file
        )


    return (
        fig,
        ax,
    )

In [ ]:
# ---------------------------------------------------------
# Patent-citation jurisdiction timeline
# Source: Lens patent-document export
# ---------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# File
# ---------------------------------------------------------

LENS_PATENT_FILE = (
    PROJECT_DIR
    / "data"
    / "lens"
    / "lenspatcite-export-mdzalfirdausi-52137fc2-30dc-4652-b8cb-a052ce9c5442-2026-09-27_06-13-31.xlsx"
)


# ---------------------------------------------------------
# Read Lens patent-document data
# ---------------------------------------------------------

lens_patents = pd.read_excel(
    LENS_PATENT_FILE
)


print("LENS PATENT DOCUMENTS")
print("=" * 50)

print(
    "Rows:",
    f"{len(lens_patents):,}"
)

print(
    "Unique Lens IDs:",
    f"{lens_patents['LensID'].nunique():,}"
)

print(
    "Unique publication keys:",
    f"{lens_patents['Patent publication key'].nunique():,}"
)

In [ ]:
# ---------------------------------------------------------
# Prepare patent citation events
# ---------------------------------------------------------

patent_events = (
    lens_patents[
        [
            "LensID",
            "Patent jurisdiction",
            "Patent publication key",
            "Patent document type",
            "Patent publication date",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Parse patent publication date
# ---------------------------------------------------------

patent_events[
    "Patent_Publication_Date"
] = pd.to_datetime(
    patent_events[
        "Patent publication date"
    ],
    errors="coerce",
)


patent_events[
    "Patent_Publication_Year"
] = (
    patent_events[
        "Patent_Publication_Date"
    ]
    .dt.year
)


# ---------------------------------------------------------
# Clean patent jurisdiction
# ---------------------------------------------------------

patent_events[
    "Patent_Jurisdiction"
] = (
    patent_events[
        "Patent jurisdiction"
    ]
    .fillna(
        "Unknown"
    )
    .astype(str)
    .str.strip()
    .str.upper()
)


patent_events.loc[
    patent_events[
        "Patent_Jurisdiction"
    ].eq(""),
    "Patent_Jurisdiction",
] = "Unknown"


# ---------------------------------------------------------
# Keep desired analysis period
# ---------------------------------------------------------

patent_events = (
    patent_events
    .loc[
        lambda df:
            df[
                "Patent_Publication_Year"
            ].between(
                2015,
                2026,
            )
    ]
    .copy()
)


# ---------------------------------------------------------
# Integer year
# ---------------------------------------------------------

patent_events[
    "Patent_Publication_Year"
] = (
    patent_events[
        "Patent_Publication_Year"
    ]
    .astype(int)
)


print()
print(
    "Patent documents during 2015-2026:",
    f"{len(patent_events):,}"
)


display(
    patent_events.head()
)

In [ ]:
# ---------------------------------------------------------
# Patent-jurisdiction distribution
# ---------------------------------------------------------

patent_jurisdiction_counts = (
    patent_events[
        "Patent_Jurisdiction"
    ]
    .value_counts()
)


display(
    patent_jurisdiction_counts
    .head(20)
    .rename_axis(
        "Patent_Jurisdiction"
    )
    .reset_index(
        name="Patent_Documents"
    )
)

In [ ]:
# ---------------------------------------------------------
# Standardize patent-jurisdiction display names
# ---------------------------------------------------------

PATENT_JURISDICTION_LABELS = {
    "US": "United States",
    "CN": "China",
    "EP": "EPO",
    "WO": "WIPO/PCT",
    "FR": "France",
    "DE": "Germany",
    "GB": "United Kingdom",
    "IT": "Italy",
    "SE": "Sweden",
}


patent_events[
    "Patent_Jurisdiction_Display"
] = (
    patent_events[
        "Patent_Jurisdiction"
    ]
    .replace(
        PATENT_JURISDICTION_LABELS
    )
)


# ---------------------------------------------------------
# Count patent jurisdictions using display names
# ---------------------------------------------------------

patent_jurisdiction_counts = (
    patent_events[
        "Patent_Jurisdiction_Display"
    ]
    .value_counts()
)


display(
    patent_jurisdiction_counts
    .head(20)
    .rename_axis(
        "Patent_Jurisdiction"
    )
    .reset_index(
        name="Patent_Documents"
    )
)


# ---------------------------------------------------------
# Leading patent jurisdictions
# ---------------------------------------------------------

TOP_N_PATENT_JURISDICTIONS = 9


top_patent_jurisdictions = (
    patent_jurisdiction_counts
    .head(
        TOP_N_PATENT_JURISDICTIONS
    )
    .index
    .tolist()
)


# ---------------------------------------------------------
# Group remaining jurisdictions as Other
# ---------------------------------------------------------

patent_events[
    "Patent_Jurisdiction_Group"
] = np.where(
    patent_events[
        "Patent_Jurisdiction_Display"
    ].isin(
        top_patent_jurisdictions
    ),
    patent_events[
        "Patent_Jurisdiction_Display"
    ],
    "Other",
)


# ---------------------------------------------------------
# Order legend by overall frequency
# ---------------------------------------------------------

patent_jurisdiction_order = (
    patent_events[
        "Patent_Jurisdiction_Group"
    ]
    .value_counts()
    .index
    .tolist()
)


patent_events[
    "Patent_Jurisdiction_Group"
] = pd.Categorical(
    patent_events[
        "Patent_Jurisdiction_Group"
    ],
    categories=patent_jurisdiction_order,
    ordered=True,
)


# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print(
    "Leading patent jurisdictions:"
)

for jurisdiction in top_patent_jurisdictions:

    print(
        " -",
        jurisdiction,
    )


print()
print(
    "Final legend order:"
)

for jurisdiction in patent_jurisdiction_order:

    print(
        " -",
        jurisdiction,
    )

In [ ]:
# ---------------------------------------------------------
# Order jurisdictions by overall frequency
# ---------------------------------------------------------

patent_jurisdiction_order = (
    patent_events[
        "Patent_Jurisdiction_Group"
    ]
    .value_counts()
    .index
    .tolist()
)


patent_events[
    "Patent_Jurisdiction_Group"
] = pd.Categorical(
    patent_events[
        "Patent_Jurisdiction_Group"
    ],
    categories=patent_jurisdiction_order,
    ordered=True,
)


display(
    patent_events[
        "Patent_Jurisdiction_Group"
    ]
    .value_counts(
        sort=False
    )
    .rename_axis(
        "Patent_Jurisdiction"
    )
    .reset_index(
        name="Patent_Documents"
    )
)

In [ ]:
# ---------------------------------------------------------
# Policy-citation jurisdiction timeline
# Source: citation-level Overton export
# ---------------------------------------------------------

from pathlib import Path
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.offsetbox import (
    OffsetImage,
    AnnotationBbox,
)


# ---------------------------------------------------------
# File
# ---------------------------------------------------------

OVERTON_CITATIONS_FILE = (
    PROJECT_DIR
    / "data"
    / "overton"
    / "Sep15 DOI based data from Overton"
    / "articles-2026-09-15.csv"
)


# ---------------------------------------------------------
# Read citation-level Overton data
# ---------------------------------------------------------

overton_citations = pd.read_csv(
    OVERTON_CITATIONS_FILE
)


print(
    "Citation rows:",
    f"{len(overton_citations):,}"
)


print(
    "Total policy citation count:",
    f"{overton_citations['Policy citation count'].sum():,}"
)

In [ ]:
# ---------------------------------------------------------
# Patent-jurisdiction donut timeline
# ---------------------------------------------------------

patent_jurisdiction_figure = (
    FIGURE_DIR
    / "patent_citation_jurisdiction_timeline.pdf"
)


fig, ax = plot_timeline_pies(
    patent_events,

    x_col=(
        "Patent_Publication_Year"
    ),

    hue_col=(
        "Patent_Jurisdiction_Group"
    ),

    # One row = one citing patent document.
    y_col=None,

    # Same visual geometry as policy figure.
    donut_scale=0.31,

    x_spacing=3.4,

    figsize=(
        18,
        6.5,
    ),

    title=(
        "Patent Citations by Jurisdiction"
        # "by Jurisdiction and Publication Year"
    ),

    y_label=(
        "Number of citing patent documents"
    ),

    legend_title=(
        "Patent jurisdiction"
    ),

    output_file=(
        patent_jurisdiction_figure
    ),
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Prepare policy citation events
# ---------------------------------------------------------

policy_events = (
    overton_citations[
        [
            "Cited by date",
            "Cited by country",
            "Cited by source",
            "Cited by organisation type",
        ]
    ]
    .copy()
)


policy_events[
    "Citation_Date"
] = pd.to_datetime(
    policy_events[
        "Cited by date"
    ],
    errors="coerce",
)


policy_events[
    "Citation_Year"
] = (
    policy_events[
        "Citation_Date"
    ]
    .dt.year
)


policy_events[
    "Policy_Jurisdiction"
] = (
    policy_events[
        "Cited by country"
    ]
    .fillna(
        "Unknown"
    )
    .astype(str)
    .str.strip()
)


policy_events.loc[
    policy_events[
        "Policy_Jurisdiction"
    ].eq(""),
    "Policy_Jurisdiction",
] = "Unknown"


# ---------------------------------------------------------
# Keep analysis years
# ---------------------------------------------------------

policy_events = (
    policy_events
    .loc[
        lambda df:
            df[
                "Citation_Year"
            ].between(
                2015,
                2026,
            )
    ]
    .copy()
)


policy_events[
    "Citation_Year"
] = (
    policy_events[
        "Citation_Year"
    ]
    .astype(int)
)


print(
    "Policy citation events:",
    f"{len(policy_events):,}"
)


display(
    policy_events.head()
)

In [ ]:
# ---------------------------------------------------------
# Standardize policy country/entity display names
# ---------------------------------------------------------

POLICY_ENTITY_LABELS = {
    "USA": "United States",
    "UK": "United Kingdom",
    "EU": "EU",
    "IGO": "IGO",
}


policy_events[
    "Policy_Entity_Display"
] = (
    policy_events[
        "Policy_Jurisdiction"
    ]
    .replace(
        POLICY_ENTITY_LABELS
    )
)


# ---------------------------------------------------------
# Count policy country/entities using display names
# ---------------------------------------------------------

policy_jurisdiction_counts = (
    policy_events[
        "Policy_Entity_Display"
    ]
    .value_counts()
)


display(
    policy_jurisdiction_counts
    .head(20)
    .rename_axis(
        "Citing_Country_Entity"
    )
    .reset_index(
        name="Policy_Citations"
    )
)


# ---------------------------------------------------------
# Leading policy countries/entities
# ---------------------------------------------------------

TOP_N_JURISDICTIONS = 9


top_policy_jurisdictions = (
    policy_jurisdiction_counts
    .head(
        TOP_N_JURISDICTIONS
    )
    .index
    .tolist()
)


# ---------------------------------------------------------
# Group remaining countries/entities as Other
# ---------------------------------------------------------

policy_events[
    "Policy_Jurisdiction_Group"
] = np.where(
    policy_events[
        "Policy_Entity_Display"
    ].isin(
        top_policy_jurisdictions
    ),
    policy_events[
        "Policy_Entity_Display"
    ],
    "Other",
)


# ---------------------------------------------------------
# Order legend by overall frequency
# ---------------------------------------------------------

policy_jurisdiction_order = (
    policy_events[
        "Policy_Jurisdiction_Group"
    ]
    .value_counts()
    .index
    .tolist()
)


policy_events[
    "Policy_Jurisdiction_Group"
] = pd.Categorical(
    policy_events[
        "Policy_Jurisdiction_Group"
    ],
    categories=policy_jurisdiction_order,
    ordered=True,
)


# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print(
    "Leading policy countries/entities:"
)

for entity in top_policy_jurisdictions:

    print(
        " -",
        entity,
    )


print()
print(
    "Final legend order:"
)

for entity in policy_jurisdiction_order:

    print(
        " -",
        entity,
    )

In [ ]:
policy_jurisdiction_figure = (
    FIGURE_DIR
    / "policy_citation_country_timeline.pdf"
)


fig, ax = plot_timeline_pies(
    policy_events,
    x_col="Citation_Year",
    hue_col="Policy_Jurisdiction_Group",

    # One row = one policy citation event,
    # therefore y_col must be None.
    y_col=None,

    donut_scale=0.31,
    x_spacing=3.4,
    figsize=(
        18,
        6.5,
    ),

    title=(
        "Policy Citations by Geographic Distribution"
        # "by Citation Year"
        ),

    y_label=(
        "Number of policy citations"
    ),

    legend_title=(
        "Cited by"
    ),

    output_file=(
        policy_jurisdiction_figure
    ),
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Policy citation events:
# organisation type and organisation sector
# ---------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Prepare citation-level data
# ---------------------------------------------------------

policy_organisation_events = (
    overton_citations[
        [
            "Cited by date",
            "Cited by organisation type",
            "Cited by organisation sector",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Parse policy-document citation date
# ---------------------------------------------------------

policy_organisation_events[
    "Citation_Date"
] = pd.to_datetime(
    policy_organisation_events[
        "Cited by date"
    ],
    errors="coerce",
)


policy_organisation_events[
    "Citation_Year"
] = (
    policy_organisation_events[
        "Citation_Date"
    ]
    .dt.year
)


# ---------------------------------------------------------
# Keep analysis period
# ---------------------------------------------------------

policy_organisation_events = (
    policy_organisation_events
    .loc[
        lambda df:
            df[
                "Citation_Year"
            ].between(
                2015,
                2026,
            )
    ]
    .copy()
)


# ---------------------------------------------------------
# Convert year to integer
# ---------------------------------------------------------

policy_organisation_events[
    "Citation_Year"
] = (
    policy_organisation_events[
        "Citation_Year"
    ]
    .astype(int)
)


# ---------------------------------------------------------
# Clean organisation type
# ---------------------------------------------------------

policy_organisation_events[
    "Organisation_Type"
] = (
    policy_organisation_events[
        "Cited by organisation type"
    ]
    .fillna(
        "Unknown"
    )
    .astype(str)
    .str.strip()
)


policy_organisation_events.loc[
    policy_organisation_events[
        "Organisation_Type"
    ].eq(""),
    "Organisation_Type",
] = "Unknown"


# ---------------------------------------------------------
# Clean organisation sector
# ---------------------------------------------------------

policy_organisation_events[
    "Organisation_Sector"
] = (
    policy_organisation_events[
        "Cited by organisation sector"
    ]
    .fillna(
        "Unknown"
    )
    .astype(str)
    .str.strip()
)


policy_organisation_events.loc[
    policy_organisation_events[
        "Organisation_Sector"
    ].eq(""),
    "Organisation_Sector",
] = "Unknown"


# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print(
    "Policy citation events:",
    f"{len(policy_organisation_events):,}"
)


print()
print(
    "ORGANISATION TYPES"
)
print(
    "=" * 50
)


display(
    policy_organisation_events[
        "Organisation_Type"
    ]
    .value_counts()
    .rename_axis(
        "Organisation_Type"
    )
    .reset_index(
        name="Policy_Citations"
    )
)


print()
print(
    "ORGANISATION SECTORS"
)
print(
    "=" * 50
)


display(
    policy_organisation_events[
        "Organisation_Sector"
    ]
    .value_counts()
    .rename_axis(
        "Organisation_Sector"
    )
    .reset_index(
        name="Policy_Citations"
    )
)

In [ ]:
# ---------------------------------------------------------
# Organisation-type ordering
# ---------------------------------------------------------

organisation_type_order = (
    policy_organisation_events[
        "Organisation_Type"
    ]
    .value_counts()
    .index
    .tolist()
)


policy_organisation_events[
    "Organisation_Type"
] = pd.Categorical(
    policy_organisation_events[
        "Organisation_Type"
    ],
    categories=organisation_type_order,
    ordered=True,
)


print(
    "Organisation type order:",
    organisation_type_order
)

In [ ]:
# ---------------------------------------------------------
# Plot policy citations by organisation type
# ---------------------------------------------------------

policy_type_figure = (
    FIGURE_DIR
    / "policy_citation_organisation_type_timeline.pdf"
)


fig, ax = plot_timeline_pies(
    policy_organisation_events,

    x_col="Citation_Year",

    hue_col="Organisation_Type",

    # One row = one policy citation event.
    y_col=None,

    # Slightly smaller donuts and greater spacing
    # prevent overlap in high-volume years.
    donut_scale=0.31,

    x_spacing=3.4,

    figsize=(
        18,
        6.5,
    ),

    title=(
        "Policy-Citation Organisation-Type "
        "Distribution by Citation Year"
    ),

    y_label=(
        "Number of policy citations"
    ),

    legend_title=(
        "Organisation type"
    ),

    output_file=(
        policy_type_figure
    ),
)


plt.show()

In [ ]:
# ---------------------------------------------------------
# Organisation-sector ordering
# ---------------------------------------------------------

preferred_sector_order = [
    "Public Sector",
    "Private Sector",
    "Third Sector",
    "Unknown",
]


available_sectors = (
    policy_organisation_events[
        "Organisation_Sector"
    ]
    .dropna()
    .unique()
    .tolist()
)


organisation_sector_order = [
    sector
    for sector
    in preferred_sector_order
    if sector
    in available_sectors
]


# Include any unexpected Overton categories
# after the expected categories.
organisation_sector_order += [
    sector
    for sector
    in available_sectors
    if sector
    not in organisation_sector_order
]


policy_organisation_events[
    "Organisation_Sector"
] = pd.Categorical(
    policy_organisation_events[
        "Organisation_Sector"
    ],
    categories=organisation_sector_order,
    ordered=True,
)


print(
    "Organisation sector order:",
    organisation_sector_order
)

In [ ]:
# ---------------------------------------------------------
# Plot policy citations by organisation sector
# ---------------------------------------------------------

policy_sector_figure = (
    FIGURE_DIR
    / "policy_citation_organisation_sector_timeline.pdf"
)


fig, ax = plot_timeline_pies(
    policy_organisation_events,

    x_col="Citation_Year",

    hue_col="Organisation_Sector",

    # One row = one policy citation event.
    y_col=None,

    donut_scale=0.31,

    x_spacing=3.4,

    figsize=(
        18,
        6.5,
    ),

    title=(
        "Policy-Citation Organisation-Sector "
        "Distribution by Citation Year"
    ),

    y_label=(
        "Number of policy citations"
    ),

    legend_title=(
        "Organisation sector"
    ),

    output_file=(
        policy_sector_figure
    ),
)


plt.show()